# exp361 exp333 candidate-path add-one novelty audit

This CPU-only train-side readout loads the saved exp333 Stage 1 OOF as one
target-free candidate path. It freezes that path, the exp293 deployable12
candidate bank, and non-overlapping block assignments before evaluation truth
is attached. Direct scores are context only; the scientific decision is the
exp302-compatible add-one novelty gate. No model, selector, blend, inference,
or submission is created.

## Contents
1. Imports and immutable contract
2. Runtime, path, SHA, and serialization helpers
3. Saved exp333 OOF candidate loader
4. Exp293 fixed candidate-bank and block helpers
5. Target-free OOF generation and freeze boundary
6. Post-freeze truth loader and direct readout
7. Add-one candidate novelty readout
8. PASS/FAIL decision and generated artifacts
9. Setup, contract preview, and execution

In [ ]:
from __future__ import annotations

import glob
import gzip
import hashlib
import json
import math
import os
from collections.abc import Iterable, Mapping, Sequence
from dataclasses import dataclass, replace
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import yaml

EXPERIMENT_NAME = "exp361_exp333_candidate_path_addone_novelty_audit"
OUTPUT_PREFIX = EXPERIMENT_NAME
KAGGLE_WORKING_ROOT = Path("/kaggle/working")
VALUE_KEY_COLUMNS = ["id", "well", "well_row_idx", "outer_fold", "md_since"]
VALUE_READ_COLUMNS = VALUE_KEY_COLUMNS + [
    "last_known_tvt",
    "candidate_tvt",
    "candidate_available",
    "candidate_finite",
]
EXPECTED_CANDIDATE_ORDER = (
    "exp226_k16",
    "selfgr_hmm_a070",
    "likpf_mean",
    "exact_hmm",
    "pf_ancc",
    "beam_mean",
    "exp226_k16__selfgr_hmm_a070",
    "exp226_k16__exact_hmm",
    "exp226_k16__likpf_mean",
    "selfgr_hmm_a070__likpf_mean",
    "likpf_mean__exact_hmm",
    "exp226_w500_50_50",
)
FORMULA_CANDIDATES = EXPECTED_CANDIDATE_ORDER[6:]
EXPECTED_VARIANTS = {"exp333_segment_offset": 16}
FORBIDDEN_CANDIDATE_COLUMNS = {
    "TVT",
    "target",
    "true_tvt",
    "tvt_true",
    "error",
    "abs_error",
    "oracle",
    "oracle_label",
    "oracle_candidate",
}
DIRECT_BUCKETS = {
    "000_050": (0.0, 50.0),
    "050_100": (50.0, 100.0),
    "100_250": (100.0, 250.0),
    "250_500": (250.0, 500.0),
    "500_1000": (500.0, 1000.0),
    "1000_plus": (1000.0, math.inf),
}

## 2. Runtime, path, SHA, and serialization helpers

In [ ]:
def in_notebook_runtime() -> bool:
    try:
        shell = get_ipython()  # type: ignore[name-defined]
    except NameError:
        return False
    return shell is not None and shell.__class__.__name__ == "ZMQInteractiveShell"


EXECUTE_NOTEBOOK = (
    os.environ.get("EXP361_IMPORT_ONLY", "0") != "1" and in_notebook_runtime()
)


def project_root() -> Path:
    start = Path.cwd()
    for candidate in (start, *start.parents):
        if (candidate / "project.yml").exists():
            return candidate
    return start


def experiment_dir() -> Path:
    root = project_root()
    nested = root / "experiments" / EXPERIMENT_NAME
    return nested if nested.exists() else Path.cwd()


def find_config_path() -> Path:
    for candidate in (
        Path.cwd() / "config.yaml",
        experiment_dir() / "config.yaml",
    ):
        if candidate.exists():
            return candidate
    matches = sorted(Path.cwd().rglob(f"{EXPERIMENT_NAME}/config.yaml"))
    if len(matches) == 1:
        return matches[0]
    raise FileNotFoundError("exp361 config.yaml was not found unambiguously")


def read_yaml(path: Path) -> dict[str, Any]:
    value = yaml.safe_load(path.read_text()) or {}
    if not isinstance(value, dict):
        raise ValueError(f"{path} must contain a YAML mapping")
    return value


def get_nested(mapping: Mapping[str, Any], dotted_key: str, default: Any = None) -> Any:
    current: Any = mapping
    for part in dotted_key.split("."):
        if not isinstance(current, Mapping) or part not in current:
            return default
        current = current[part]
    return current


def runtime_artifacts_dir() -> Path:
    path = (
        KAGGLE_WORKING_ROOT / "artifacts"
        if KAGGLE_WORKING_ROOT.exists()
        else experiment_dir() / "artifacts"
    )
    path.mkdir(parents=True, exist_ok=True)
    return path


def runtime_work_dir() -> Path:
    path = (
        KAGGLE_WORKING_ROOT / ".exp361_work"
        if KAGGLE_WORKING_ROOT.exists()
        else experiment_dir() / ".exp361_work"
    )
    path.mkdir(parents=True, exist_ok=True)
    return path


def runtime_metrics_path() -> Path:
    return (
        KAGGLE_WORKING_ROOT / "metrics.json"
        if KAGGLE_WORKING_ROOT.exists()
        else experiment_dir() / "metrics.json"
    )


def sha256_file(path: Path, chunk_bytes: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(chunk_bytes):
            digest.update(block)
    return digest.hexdigest()


def sha256_decompressed_gzip(path: Path, chunk_bytes: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with gzip.open(path, "rb") as handle:
        while block := handle.read(chunk_bytes):
            digest.update(block)
    return digest.hexdigest()


def json_sha256(value: Any) -> str:
    payload = json.dumps(
        to_jsonable(value),
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ).encode()
    return hashlib.sha256(payload).hexdigest()


def to_jsonable(value: Any) -> Any:
    if isinstance(value, Mapping):
        return {str(key): to_jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value


def _normalize_frame_for_hash(frame: pd.DataFrame) -> pd.DataFrame:
    string_columns = [
        column
        for column, dtype in frame.dtypes.items()
        if isinstance(dtype, pd.StringDtype)
    ]
    if not string_columns:
        return frame
    normalized = frame.copy()
    for column in string_columns:
        normalized[column] = normalized[column].astype(object)
    return normalized


def frame_content_sha256(
    frame: pd.DataFrame, columns: Iterable[str] | None = None
) -> str:
    selected = frame if columns is None else frame[list(columns)]
    selected = _normalize_frame_for_hash(selected)
    digest = hashlib.sha256()
    digest.update("|".join(selected.columns).encode())
    digest.update("|".join(str(dtype) for dtype in selected.dtypes).encode())
    row_hashes = pd.util.hash_pandas_object(selected, index=False, categorize=True)
    digest.update(
        row_hashes.to_numpy(dtype="uint64").astype("<u8", copy=False).tobytes()
    )
    return digest.hexdigest()


def frame_schema_sha256(frame: pd.DataFrame) -> str:
    normalized = _normalize_frame_for_hash(frame)
    schema = [(column, str(dtype)) for column, dtype in normalized.dtypes.items()]
    return hashlib.sha256(
        json.dumps(schema, separators=(",", ":")).encode()
    ).hexdigest()


def write_json(path: Path, value: Mapping[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(to_jsonable(value), indent=2, sort_keys=True, ensure_ascii=False)
        + "\n"
    )


def _write_frame(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False, float_format="%.12g", lineterminator="\n")


def write_gzip_frame(path: Path, frame: pd.DataFrame) -> tuple[str, str]:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(
        path,
        index=False,
        float_format="%.12g",
        lineterminator="\n",
        compression={"method": "gzip", "compresslevel": 1, "mtime": 0},
    )
    return sha256_file(path), sha256_decompressed_gzip(path)


def expand_existing_paths(patterns: Sequence[str]) -> list[Path]:
    found: list[Path] = []
    root = project_root()
    for raw_pattern in patterns:
        raw = str(raw_pattern)
        path = Path(raw)
        direct = path if path.is_absolute() else root / path
        if direct.exists():
            found.append(direct)
            continue
        for match in glob.glob(raw, recursive=True):
            candidate = Path(match)
            if candidate.exists():
                found.append(candidate)
        if not path.is_absolute():
            for match in glob.glob(str(root / raw), recursive=True):
                candidate = Path(match)
                if candidate.exists():
                    found.append(candidate)
    unique: dict[str, Path] = {}
    for path in found:
        unique.setdefault(str(path.resolve()), path)
    return list(unique.values())


def resolve_file(
    patterns: Sequence[str],
    *,
    label: str,
    expected_sha256: str | None = None,
) -> Path:
    candidates = [path for path in expand_existing_paths(patterns) if path.is_file()]
    if expected_sha256:
        matching = [path for path in candidates if sha256_file(path) == expected_sha256]
        if matching:
            return sorted(matching, key=lambda item: len(str(item)))[0]
        if candidates:
            evidence = {str(path): sha256_file(path) for path in candidates}
            raise ValueError(f"{label} SHA mismatch: {evidence}")
    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise FileNotFoundError(f"{label} not found from patterns: {patterns}")
    raise ValueError(f"{label} resolved to multiple files: {candidates}")


def resolve_gzip_by_decompressed_sha(
    patterns: Sequence[str], *, label: str, expected_sha256: str
) -> Path:
    candidates = [path for path in expand_existing_paths(patterns) if path.is_file()]
    matches = [
        path for path in candidates if sha256_decompressed_gzip(path) == expected_sha256
    ]
    if matches:
        return sorted(matches, key=lambda item: len(str(item)))[0]
    evidence = {
        str(path): sha256_decompressed_gzip(path) for path in candidates
    }
    if evidence:
        raise ValueError(f"{label} decompressed SHA mismatch: {evidence}")
    raise FileNotFoundError(f"{label} not found from patterns: {patterns}")


def reject_forbidden_candidate_columns(columns: Iterable[str]) -> None:
    normalized = {str(column) for column in columns}
    forbidden = normalized & FORBIDDEN_CANDIDATE_COLUMNS
    token_forbidden = {
        column
        for column in normalized
        if any(
            token in column.lower()
            for token in ("true_tvt", "tvt_true", "abs_error", "oracle_label")
        )
    }
    if forbidden or token_forbidden:
        raise ValueError(
            "candidate partition exposes forbidden truth/readout columns: "
            f"{sorted(forbidden | token_forbidden)}"
        )

## 3. Saved exp333 OOF candidate loader

The completed exp333 Stage 1 OOF is loaded through a target-free allowlist.
The fixed file, decompressed-content, logical-prediction, row-identity, and
outer-fold contracts are checked before the aligned candidate is frozen.

In [ ]:
@dataclass
class VariantPrediction:
    variant: str
    k_segments: int
    values: np.ndarray
    frame: pd.DataFrame
    kappa_by_fold: pd.DataFrame
    prediction_content_sha256: str
    donor_fit_truth_wells: int
    fold_run_count: int
    source_evidence: dict[str, Any] | None = None


def resolve_raw_train_dir(
    config: Mapping[str, Any], expected_wells: set[str]
) -> tuple[Path, list[Path]]:
    patterns = list(get_nested(config, "data.raw_train_dir_patterns"))
    horizontal_glob = str(get_nested(config, "data.raw_horizontal_glob"))
    candidates = [path for path in expand_existing_paths(patterns) if path.is_dir()]
    evidence: list[tuple[Path, list[Path], set[str]]] = []
    for directory in candidates:
        files = sorted(directory.glob(horizontal_glob))
        wells = {
            path.name.split("__horizontal_well.csv", 1)[0] for path in files
        }
        evidence.append((directory, files, wells))
        if wells == expected_wells and len(files) == len(expected_wells):
            return directory, files
    detail = {
        str(directory): {"files": len(files), "wells": len(wells)}
        for directory, files, wells in evidence
    }
    raise FileNotFoundError(
        "raw train directory with exact candidate-well inventory was not found: "
        f"{detail}"
    )


def build_raw_input_manifest(
    raw_dir: Path, horizontal_files: Sequence[Path]
) -> pd.DataFrame:
    records: list[dict[str, Any]] = []
    for horizontal_path in horizontal_files:
        well = horizontal_path.name.split("__horizontal_well.csv", 1)[0]
        typewell_path = horizontal_path.with_name(f"{well}__typewell.csv")
        for role, path in (
            ("horizontal", horizontal_path),
            ("typewell", typewell_path),
        ):
            if not path.exists():
                raise FileNotFoundError(f"raw {role} file missing: {path}")
            records.append(
                {
                    "well": well,
                    "role": role,
                    "relative_path": str(path.relative_to(raw_dir)),
                    "bytes": path.stat().st_size,
                    "file_sha256": sha256_file(path),
                }
            )
    return pd.DataFrame(records).sort_values(
        ["well", "role"], kind="stable"
    ).reset_index(drop=True)


def exp333_saved_prediction_sha256(frame: pd.DataFrame) -> str:
    """Reproduce the logical OOF hash written by the completed exp333 run."""
    columns = ["well_id", "row_idx", "outer_fold", "tvt_pred_stage1"]
    selected = frame.loc[:, columns]
    header = json.dumps(
        {"columns": columns},
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=True,
    ).encode()
    digest = hashlib.sha256(header)
    digest.update(
        pd.util.hash_pandas_object(selected, index=False)
        .to_numpy(np.uint64)
        .tobytes()
    )
    return digest.hexdigest()


def build_variant_oof(
    variant: str,
    config: Mapping[str, Any],
    bank: CandidateBank,
    horizontal_files: Sequence[Path],
    expected_fold_by_well: Mapping[str, int],
) -> VariantPrediction:
    """Load the saved exp333 OOF without opening its colocated truth columns."""
    del horizontal_files
    if variant != "exp333_segment_offset":
        raise ValueError(f"unexpected readout candidate: {variant}")
    source_cfg = get_nested(config, "data.exp333_oof")
    path = resolve_gzip_by_decompressed_sha(
        source_cfg["patterns"],
        label="saved exp333 Stage 1 OOF",
        expected_sha256=str(source_cfg["expected_decompressed_sha256"]),
    )
    file_sha = sha256_file(path)
    if file_sha != str(source_cfg["expected_file_sha256"]):
        raise ValueError(
            "saved exp333 Stage 1 OOF file SHA mismatch: "
            f"{file_sha} != {source_cfg['expected_file_sha256']}"
        )
    allowlist = list(source_cfg["pre_freeze_columns"])
    frame = pd.read_csv(path, usecols=allowlist)
    if set(frame.columns) != set(allowlist):
        raise ValueError(f"exp333 target-free allowlist mismatch: {list(frame.columns)}")
    forbidden = set(source_cfg["forbidden_pre_freeze_columns"]) & set(frame.columns)
    if forbidden:
        raise ValueError(f"exp333 pre-freeze truth/error columns loaded: {sorted(forbidden)}")
    if len(frame) != int(get_nested(config, "validation.expected_rows")):
        raise ValueError("exp333 saved OOF row count mismatch")
    if frame[["well_id", "row_idx"]].duplicated().any():
        raise ValueError("exp333 saved OOF contains duplicate row identities")

    source_logical_sha = exp333_saved_prediction_sha256(frame)
    generated = pd.DataFrame(
        {
            "id": (
                frame["well_id"].astype(str)
                + "_"
                + frame["row_idx"].astype(int).astype(str)
            ),
            "well": frame["well_id"].astype(str),
            "well_row_idx": pd.to_numeric(
                frame["row_idx"], errors="raise"
            ).to_numpy(dtype=np.int32),
            "outer_fold": pd.to_numeric(
                frame["outer_fold"], errors="raise"
            ).to_numpy(dtype=np.int8),
            "variant": variant,
            "candidate_tvt": pd.to_numeric(
                frame["tvt_pred_stage1"], errors="raise"
            ).to_numpy(dtype=np.float64),
        }
    )
    indexer = pd.Index(generated["id"]).get_indexer(bank.keys["id"].astype(str))
    if np.any(indexer < 0) or len(generated) != len(bank.keys):
        raise ValueError("exp333 identity coverage differs from the fixed bank")
    aligned = generated.iloc[indexer].reset_index(drop=True)
    for column in ("id", "well", "well_row_idx"):
        if not np.array_equal(
            aligned[column].to_numpy(), bank.keys[column].to_numpy()
        ):
            raise ValueError(f"exp333 aligned key mismatch: {column}")
    canonical_fold = {
        str(well): int(fold) for well, fold in expected_fold_by_well.items()
    }
    expected_row_folds = aligned["well"].map(canonical_fold)
    if expected_row_folds.isna().any():
        raise ValueError("exp333 contains a well outside the exp226 fold contract")
    if not np.array_equal(
        aligned["outer_fold"].to_numpy(dtype=np.int64),
        expected_row_folds.to_numpy(dtype=np.int64),
    ):
        raise ValueError("exp333 outer folds differ from saved exp226 folds")
    values = aligned["candidate_tvt"].to_numpy(dtype=np.float64)
    if not np.isfinite(values).all():
        raise ValueError("exp333 saved OOF contains nonfinite predictions")
    output_frame = aligned[
        ["id", "well", "well_row_idx", "outer_fold", "variant", "candidate_tvt"]
    ].copy()
    content_sha = frame_content_sha256(output_frame)
    fold_readout = (
        output_frame.groupby("outer_fold", sort=True)
        .size()
        .rename("rows")
        .reset_index()
    )
    fold_readout["source"] = "saved_exp333_stage1_oof"
    fold_readout["model_configs"] = 0
    fold_readout["trained_folds"] = 0
    fold_readout["boosters"] = 0
    return VariantPrediction(
        variant=variant,
        k_segments=16,
        values=values,
        frame=output_frame,
        kappa_by_fold=fold_readout,
        prediction_content_sha256=content_sha,
        donor_fit_truth_wells=0,
        fold_run_count=int(fold_readout["outer_fold"].nunique()),
        source_evidence={
            "phase": "target_free_saved_candidate",
            "source": "exp333_stage1_oof_allowlist",
            "path": str(path),
            "rows": len(frame),
            "file_sha256": file_sha,
            "decompressed_content_sha256": sha256_decompressed_gzip(path),
            "logical_content_sha256": source_logical_sha,
            "upstream_reported_prediction_sha256": str(
                source_cfg["upstream_reported_prediction_sha256"]
            ),
            "schema_sha256": frame_schema_sha256(frame),
        },
    )

## 4. Exp293 fixed candidate-bank and block helpers

The bank is reconstructed from the exp263 manifest with the exact float32
formula order used by exp293. Its content SHA and the decompressed block
assignment SHA must equal the completed exp293 run before exp333 can be
evaluated as an add-one candidate.

In [ ]:
@dataclass
class CandidateBank:
    keys: pd.DataFrame
    candidate_ids: tuple[str, ...]
    values: np.memmap
    values_path: Path
    primitive_ids: tuple[str, ...]
    manifest: dict[str, Any]
    manifest_path: Path
    key_content_sha256: str
    candidate_content_sha256: str
    coverage_by_candidate: dict[str, float]
    sample_parity: pd.DataFrame
    input_evidence: list[dict[str, Any]]


def _artifact_path_from_manifest(
    manifest_path: Path, item: Mapping[str, Any]
) -> Path:
    raw = str(item["path"])
    marker = "/artifacts/"
    if marker in raw:
        relative = raw.split(marker, 1)[1]
        candidate = manifest_path.parent / relative
        if candidate.exists():
            return candidate
    direct = Path(raw)
    if direct.exists():
        return direct
    suffix = Path(raw).parts[-4:]
    for root in manifest_path.parents:
        candidate = root.joinpath(*suffix)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"manifest partition is missing: {raw}")


def _read_manifest_partitions(
    manifest_path: Path,
    items: Sequence[Mapping[str, Any]],
    *,
    columns: Sequence[str],
    label: str,
) -> tuple[pd.DataFrame, list[dict[str, Any]]]:
    frames: list[pd.DataFrame] = []
    evidence: list[dict[str, Any]] = []
    for item in items:
        path = _artifact_path_from_manifest(manifest_path, item)
        actual_file_sha = sha256_file(path)
        expected_file_sha = str(item.get("file_sha256", ""))
        if expected_file_sha and actual_file_sha != expected_file_sha:
            raise ValueError(f"{label} partition file SHA mismatch: {path}")
        full = pd.read_parquet(path)
        reject_forbidden_candidate_columns(full.columns)
        expected_rows = int(item.get("rows", len(full)))
        if len(full) != expected_rows:
            raise ValueError(f"{label} partition row mismatch: {path}")
        expected_schema = str(item.get("schema_sha256", ""))
        actual_schema = frame_schema_sha256(full)
        if expected_schema and actual_schema != expected_schema:
            raise ValueError(f"{label} partition schema SHA mismatch: {path}")
        expected_content = str(item.get("content_sha256", ""))
        actual_content = frame_content_sha256(full)
        if expected_content and actual_content != expected_content:
            raise ValueError(f"{label} partition content SHA mismatch: {path}")
        missing = set(columns) - set(full.columns)
        if missing:
            raise ValueError(f"{label} partition columns missing: {sorted(missing)}")
        frames.append(full[list(columns)].copy())
        evidence.append(
            {
                "phase": "target_free",
                "source": label,
                "path": str(path),
                "rows": len(full),
                "file_sha256": actual_file_sha,
                "decompressed_content_sha256": None,
                "logical_content_sha256": actual_content,
                "schema_sha256": actual_schema,
            }
        )
    if not frames:
        raise ValueError(f"no partitions declared for {label}")
    return pd.concat(frames, ignore_index=True), evidence


def _assert_same_keys(
    reference: pd.DataFrame, candidate: pd.DataFrame, label: str
) -> None:
    if len(reference) != len(candidate):
        raise ValueError(f"{label} key row count mismatch")
    for column in VALUE_KEY_COLUMNS:
        left = reference[column].to_numpy()
        right = candidate[column].to_numpy()
        equal = (
            np.array_equal(left, right, equal_nan=True)
            if column == "md_since"
            else np.array_equal(left, right)
        )
        if not equal:
            raise ValueError(f"{label} key mismatch in {column}")


def candidate_bank_content_sha256(
    bank: CandidateBank, chunk_rows: int = 100_000
) -> str:
    digest = hashlib.sha256()
    digest.update(json.dumps(list(bank.candidate_ids), separators=(",", ":")).encode())
    digest.update(bank.key_content_sha256.encode())
    for position, candidate_id in enumerate(bank.candidate_ids):
        digest.update(candidate_id.encode())
        for start in range(0, len(bank.keys), chunk_rows):
            end = min(start + chunk_rows, len(bank.keys))
            values = np.asarray(bank.values[start:end, position], dtype="<f4")
            digest.update(values.tobytes())
    return digest.hexdigest()


def _materialize_formulas(
    values: np.memmap,
    column_by_candidate: Mapping[str, int],
    config: Mapping[str, Any],
) -> None:
    pairs = get_nested(config, "candidate_bank.pairs")
    for candidate_id, weights in pairs.items():
        parents = list(weights)
        if len(parents) != 2 or any(
            not math.isclose(float(weights[parent]), 0.5) for parent in parents
        ):
            raise ValueError(f"{candidate_id} differs from fixed 50/50 contract")
        left = values[:, column_by_candidate[parents[0]]]
        right = values[:, column_by_candidate[parents[1]]]
        values[:, column_by_candidate[str(candidate_id)]] = (
            np.float32(0.5) * (left + right)
        ).astype(np.float32)

    fixed = get_nested(config, "candidate_bank.fixed_formula")
    if list(fixed) != ["exp226_w500_50_50"]:
        raise ValueError("fixed formula identity differs from exp293 contract")
    weights = fixed["exp226_w500_50_50"]
    expected = {"exp226_k16": 0.5, "likpf_mean": 0.25, "exact_hmm": 0.25}
    if weights != expected:
        raise ValueError("exp226_w500_50_50 weights differ from fixed contract")
    output = (
        np.float32(0.5) * values[:, column_by_candidate["exp226_k16"]]
        + np.float32(0.25) * values[:, column_by_candidate["likpf_mean"]]
        + np.float32(0.25) * values[:, column_by_candidate["exact_hmm"]]
    ).astype(np.float32)
    values[:, column_by_candidate["exp226_w500_50_50"]] = output
    values.flush()


def _build_sample_parity(
    bank: CandidateBank, parity_path: Path, tolerance: float
) -> tuple[pd.DataFrame, dict[str, Any]]:
    sample = pd.read_parquet(parity_path)
    if "id" not in sample:
        raise ValueError("exp263 small parity sample lacks id")
    missing = set(FORMULA_CANDIDATES) - set(sample.columns)
    if missing:
        raise ValueError(f"exp263 small parity formulas missing: {sorted(missing)}")
    indexer = pd.Index(bank.keys["id"].astype(str)).get_indexer(
        sample["id"].astype(str)
    )
    if np.any(indexer < 0):
        raise ValueError("exp263 small parity IDs are absent from candidate bank")
    position = {name: idx for idx, name in enumerate(bank.candidate_ids)}
    records: list[dict[str, Any]] = []
    for candidate_id in FORMULA_CANDIDATES:
        actual = np.asarray(bank.values[indexer, position[candidate_id]], dtype=np.float64)
        expected = pd.to_numeric(sample[candidate_id], errors="raise").to_numpy(
            dtype=np.float64
        )
        max_abs = float(np.max(np.abs(actual - expected), initial=0.0))
        records.append(
            {
                "check_type": "exp263_small_parity_max_abs_ft",
                "candidate_id": candidate_id,
                "actual": max_abs,
                "expected": 0.0,
                "absolute_difference": max_abs,
                "tolerance": tolerance,
                "passed": bool(max_abs <= tolerance),
            }
        )
    evidence = {
        "phase": "target_free",
        "source": "exp263_small_parity_sample",
        "path": str(parity_path),
        "rows": len(sample),
        "file_sha256": sha256_file(parity_path),
        "decompressed_content_sha256": None,
        "logical_content_sha256": frame_content_sha256(sample),
        "schema_sha256": frame_schema_sha256(sample),
    }
    return pd.DataFrame(records), evidence


def build_candidate_bank(
    config: Mapping[str, Any], work_dir: Path
) -> CandidateBank:
    manifest_cfg = get_nested(config, "data.exp263_manifest")
    manifest_path = resolve_file(
        manifest_cfg["patterns"],
        label="exp263 cache manifest",
        expected_sha256=str(manifest_cfg["expected_file_sha256"]),
    )
    manifest = json.loads(manifest_path.read_text())
    expected_rows = int(get_nested(config, "validation.expected_rows"))
    expected_wells = int(get_nested(config, "validation.expected_wells"))
    expected_folds = int(get_nested(config, "validation.n_folds"))
    if int(manifest.get("rows", -1)) != expected_rows:
        raise ValueError("exp263 manifest row contract mismatch")
    if int(manifest.get("wells", -1)) != expected_wells:
        raise ValueError("exp263 manifest well contract mismatch")
    if int(manifest.get("folds", -1)) != expected_folds:
        raise ValueError("exp263 manifest fold contract mismatch")
    if manifest.get("canonical_id_sha256") != manifest_cfg[
        "expected_canonical_id_sha256"
    ]:
        raise ValueError("exp263 canonical ID SHA mismatch")

    candidate_ids = tuple(get_nested(config, "candidate_bank.order"))
    if candidate_ids != EXPECTED_CANDIDATE_ORDER:
        raise ValueError("candidate order differs from exp293 fixed contract")
    if len(candidate_ids) != int(get_nested(config, "candidate_bank.expected_count")):
        raise ValueError("candidate count differs from exp293 fixed contract")
    primitive_ids = tuple(get_nested(config, "candidate_bank.primitives"))
    if primitive_ids != EXPECTED_CANDIDATE_ORDER[:6]:
        raise ValueError("primitive order differs from exp293 fixed contract")

    values_path = work_dir / f"{OUTPUT_PREFIX}_candidate_bank.f32"
    values = np.memmap(
        values_path,
        mode="w+",
        dtype="float32",
        shape=(expected_rows, len(candidate_ids)),
    )
    values[:] = np.nan
    column_by_candidate = {name: idx for idx, name in enumerate(candidate_ids)}
    input_evidence: list[dict[str, Any]] = [
        {
            "phase": "target_free",
            "source": "exp263_manifest",
            "path": str(manifest_path),
            "rows": expected_rows,
            "file_sha256": sha256_file(manifest_path),
            "decompressed_content_sha256": None,
            "logical_content_sha256": manifest.get("canonical_id_sha256"),
            "schema_sha256": manifest.get("generation_config_sha256"),
        }
    ]
    reference_keys: pd.DataFrame | None = None
    coverage_by_candidate: dict[str, float] = {}
    for candidate_id in primitive_ids:
        items = manifest["candidate_value_partitions"].get(candidate_id)
        if not items or len(items) != expected_folds:
            raise ValueError(f"{candidate_id} must have five value partitions")
        frame, evidence = _read_manifest_partitions(
            manifest_path,
            items,
            columns=VALUE_READ_COLUMNS,
            label=f"exp263_value::{candidate_id}",
        )
        input_evidence.extend(evidence)
        if reference_keys is None:
            reference_keys = frame[VALUE_KEY_COLUMNS].copy()
            reference_keys["id"] = reference_keys["id"].astype(str)
            reference_keys["well"] = reference_keys["well"].astype(str)
        else:
            _assert_same_keys(reference_keys, frame, candidate_id)
        available = frame["candidate_available"].astype(bool).to_numpy()
        finite_flag = frame["candidate_finite"].astype(bool).to_numpy()
        candidate_values = pd.to_numeric(
            frame["candidate_tvt"], errors="coerce"
        ).to_numpy(dtype=np.float32)
        valid = available & finite_flag & np.isfinite(candidate_values)
        candidate_values[~valid] = np.nan
        values[:, column_by_candidate[candidate_id]] = candidate_values
        coverage_by_candidate[candidate_id] = float(valid.mean())

    if reference_keys is None:
        raise AssertionError("primitive candidate loading produced no keys")
    if len(reference_keys) != expected_rows:
        raise ValueError("candidate bank total row mismatch")
    if reference_keys["well"].nunique() != expected_wells:
        raise ValueError("candidate bank total well mismatch")
    if reference_keys["id"].duplicated().any():
        raise ValueError("candidate bank IDs must be unique")
    if set(reference_keys["outer_fold"].unique()) != set(range(expected_folds)):
        raise ValueError("candidate bank outer-fold coverage mismatch")

    _materialize_formulas(values, column_by_candidate, config)
    for candidate_id in candidate_ids[6:]:
        finite = np.isfinite(values[:, column_by_candidate[candidate_id]])
        coverage_by_candidate[candidate_id] = float(finite.mean())
    if any(not math.isclose(value, 1.0) for value in coverage_by_candidate.values()):
        raise ValueError(f"candidate finite coverage is not 1.0: {coverage_by_candidate}")

    key_hash = frame_content_sha256(reference_keys[VALUE_KEY_COLUMNS])
    bank = CandidateBank(
        keys=reference_keys.reset_index(drop=True),
        candidate_ids=candidate_ids,
        values=values,
        values_path=values_path,
        primitive_ids=primitive_ids,
        manifest=manifest,
        manifest_path=manifest_path,
        key_content_sha256=key_hash,
        candidate_content_sha256="",
        coverage_by_candidate=coverage_by_candidate,
        sample_parity=pd.DataFrame(),
        input_evidence=input_evidence,
    )
    bank.candidate_content_sha256 = candidate_bank_content_sha256(
        bank, int(get_nested(config, "audit.work_chunk_rows"))
    )
    parity_path = manifest_path.parent / str(
        manifest_cfg["small_parity_filename"]
    )
    if not parity_path.exists():
        raise FileNotFoundError(f"exp263 small parity sample missing: {parity_path}")
    sample_parity, parity_evidence = _build_sample_parity(
        bank,
        parity_path,
        float(get_nested(config, "candidate_bank.formula_parity_max_abs_ft")),
    )
    if not bool(sample_parity["passed"].all()):
        raise ValueError("exp263 formula sample parity failed")
    bank.sample_parity = sample_parity
    bank.input_evidence.append(parity_evidence)
    return bank


@dataclass
class GroupLayout:
    name: str
    codes: np.ndarray
    n_groups: int
    group_rows: np.ndarray
    group_well: np.ndarray
    group_fold: np.ndarray


@dataclass
class BlockAssignments:
    frame: pd.DataFrame
    well_names: np.ndarray
    well_codes: np.ndarray
    well_fold: np.ndarray
    layouts: dict[str, GroupLayout]


def assignments_with_evaluation_folds(
    assignments: BlockAssignments, row_folds: np.ndarray
) -> BlockAssignments:
    """Keep frozen exp293 block IDs but stratify readouts by exp226 folds."""
    folds = np.asarray(row_folds, dtype=np.int8)
    if len(folds) != len(assignments.well_codes):
        raise ValueError("evaluation fold row count mismatch")
    if set(np.unique(folds).astype(int)) != set(range(5)):
        raise ValueError("evaluation fold inventory mismatch")

    n_wells = len(assignments.well_names)
    first_well = np.full(n_wells, len(folds), dtype=np.int64)
    np.minimum.at(
        first_well,
        assignments.well_codes,
        np.arange(len(folds), dtype=np.int64),
    )
    well_fold = folds[first_well]
    if not np.array_equal(well_fold[assignments.well_codes], folds):
        raise ValueError("one well spans multiple evaluation folds")

    layouts: dict[str, GroupLayout] = {}
    for name, layout in assignments.layouts.items():
        first_group = np.full(layout.n_groups, len(folds), dtype=np.int64)
        np.minimum.at(
            first_group,
            layout.codes,
            np.arange(len(folds), dtype=np.int64),
        )
        layouts[name] = replace(
            layout,
            group_fold=folds[first_group].astype(np.int8),
        )
    return BlockAssignments(
        frame=assignments.frame,
        well_names=assignments.well_names,
        well_codes=assignments.well_codes,
        well_fold=well_fold,
        layouts=layouts,
    )


def _layout_from_codes(
    name: str,
    codes: np.ndarray,
    well_codes: np.ndarray,
    row_folds: np.ndarray,
) -> GroupLayout:
    if len(codes) == 0 or np.any(codes < 0):
        raise ValueError(f"invalid group codes for {name}")
    n_groups = int(codes.max()) + 1
    rows = np.bincount(codes, minlength=n_groups).astype(np.int64)
    first = np.full(n_groups, len(codes), dtype=np.int64)
    np.minimum.at(first, codes, np.arange(len(codes), dtype=np.int64))
    if np.any(first == len(codes)):
        raise ValueError(f"group code gap for {name}")
    group_well = well_codes[first].astype(np.int32)
    group_fold = row_folds[first].astype(np.int8)
    return GroupLayout(
        name=name,
        codes=codes.astype(np.int32, copy=False),
        n_groups=n_groups,
        group_rows=rows,
        group_well=group_well,
        group_fold=group_fold,
    )


def build_block_assignments(
    keys: pd.DataFrame, horizons: Sequence[int]
) -> BlockAssignments:
    wells = keys["well"].astype(str).to_numpy()
    row_folds = pd.to_numeric(keys["outer_fold"], errors="raise").to_numpy(
        dtype=np.int8
    )
    if len(wells) == 0:
        raise ValueError("candidate keys are empty")
    segment_start_mask = np.r_[True, wells[1:] != wells[:-1]]
    starts = np.flatnonzero(segment_start_mask)
    ends = np.r_[starts[1:], len(wells)]
    segment_wells = wells[starts]
    if pd.Index(segment_wells).duplicated().any():
        raise ValueError("well rows are not contiguous in candidate bank")
    lengths = (ends - starts).astype(np.int64)
    well_codes = np.repeat(
        np.arange(len(starts), dtype=np.int32), lengths
    )
    within_well = np.arange(len(wells), dtype=np.int64) - np.repeat(starts, lengths)
    well_fold = row_folds[starts]
    for start, end, fold in zip(starts, ends, well_fold, strict=True):
        if not np.all(row_folds[start:end] == fold):
            raise ValueError("one well spans multiple outer folds")
    layouts: dict[str, GroupLayout] = {}
    for horizon in horizons:
        if int(horizon) <= 0:
            raise ValueError("block horizon must be positive")
        blocks_per_well = (lengths + int(horizon) - 1) // int(horizon)
        offsets = np.r_[0, np.cumsum(blocks_per_well[:-1])].astype(np.int64)
        codes = offsets[well_codes] + within_well // int(horizon)
        name = f"h{int(horizon)}"
        layouts[name] = _layout_from_codes(
            name, codes.astype(np.int32), well_codes, row_folds
        )
    layouts["whole_well"] = _layout_from_codes(
        "whole_well", well_codes.copy(), well_codes, row_folds
    )
    assignment = keys[VALUE_KEY_COLUMNS].copy()
    assignment["well_code"] = well_codes
    for name, layout in layouts.items():
        assignment[f"{name}_group"] = layout.codes
    return BlockAssignments(
        frame=assignment,
        well_names=np.asarray(segment_wells, dtype=object),
        well_codes=well_codes,
        well_fold=well_fold,
        layouts=layouts,
    )


def load_hidden_like_sets(
    config: Mapping[str, Any], expected_wells: set[str]
) -> tuple[dict[str, set[str]], dict[str, Any]]:
    hidden_cfg = get_nested(config, "data.hidden_like_assignment")
    path = resolve_file(
        hidden_cfg["patterns"],
        label="hidden-like assignment",
        expected_sha256=str(hidden_cfg["expected_file_sha256"]),
    )
    frame = pd.read_csv(path)
    well_column = str(hidden_cfg["well_column"])
    if well_column not in frame:
        raise ValueError(f"hidden-like well column missing: {well_column}")
    sets: dict[str, set[str]] = {}
    for scope, role_column in hidden_cfg["role_columns"].items():
        if role_column not in frame:
            raise ValueError(f"hidden-like role column missing: {role_column}")
        selected = set(
            frame.loc[frame[role_column].eq("valid"), well_column].astype(str)
        )
        unknown = selected - expected_wells
        if unknown:
            raise ValueError(f"hidden-like scope has unknown wells: {sorted(unknown)[:5]}")
        sets[str(scope)] = selected
    evidence = {
        "phase": "target_free",
        "source": "hidden_like_assignment",
        "path": str(path),
        "rows": len(frame),
        "file_sha256": sha256_file(path),
        "decompressed_content_sha256": None,
        "logical_content_sha256": frame_content_sha256(frame),
        "schema_sha256": frame_schema_sha256(frame),
    }
    return sets, evidence

## 5. Target-free OOF generation and freeze boundary

The saved K16 control is opened with an explicit target-free allowlist. The
saved exp333 candidate, exp293 bank identity, block layout, raw-input
manifest, and fold readout are written and hashed before evaluation truth is
loaded by the next section.

In [ ]:
@dataclass
class SavedControl:
    path: Path
    values: np.ndarray
    frame: pd.DataFrame
    file_sha256: str
    decompressed_sha256: str
    parity_max_abs_ft: float
    folds: np.ndarray
    fold_by_well: dict[str, int]
    bank_fold_match_fraction: float


def extract_saved_control_folds(
    aligned: pd.DataFrame,
    bank: CandidateBank,
    expected_folds: int,
) -> tuple[np.ndarray, dict[str, int], float]:
    folds = pd.to_numeric(aligned["fold"], errors="raise").to_numpy(
        dtype=np.int8
    )
    if set(np.unique(folds).astype(int)) != set(range(expected_folds)):
        raise ValueError("saved control fold inventory mismatch")
    fold_frame = pd.DataFrame(
        {
            "well": aligned["well_id"].astype(str),
            "fold": folds,
        }
    )
    if bool(fold_frame.groupby("well", sort=False)["fold"].nunique().gt(1).any()):
        raise ValueError("saved control well spans multiple folds")
    fold_by_well = (
        fold_frame.drop_duplicates("well").set_index("well")["fold"].astype(int).to_dict()
    )
    if set(fold_by_well) != set(bank.keys["well"].astype(str)):
        raise ValueError("saved control fold well coverage mismatch")
    bank_folds = bank.keys["outer_fold"].to_numpy(dtype=np.int8)
    match_fraction = float(np.mean(folds == bank_folds))
    return folds, fold_by_well, match_fraction


def load_saved_control(
    config: Mapping[str, Any], bank: CandidateBank
) -> SavedControl:
    control_cfg = get_nested(config, "data.exp226_oof")
    path = resolve_gzip_by_decompressed_sha(
        control_cfg["patterns"],
        label="saved exp226 K16 OOF",
        expected_sha256=str(control_cfg["expected_decompressed_sha256"]),
    )
    allowlist = list(control_cfg["pre_freeze_columns"])
    frame = pd.read_csv(path, usecols=allowlist)
    required = {"well_id", "row_idx", "suffix_offset", "tvt_pred", "fold"}
    if set(frame.columns) != required:
        raise ValueError(f"saved control allowlist mismatch: {list(frame.columns)}")
    frame["id"] = (
        frame["well_id"].astype(str)
        + "_"
        + frame["row_idx"].astype(int).astype(str)
    )
    if frame["id"].duplicated().any():
        raise ValueError("saved control contains duplicate IDs")
    indexer = pd.Index(frame["id"].astype(str)).get_indexer(
        bank.keys["id"].astype(str)
    )
    if np.any(indexer < 0) or len(frame) != len(bank.keys):
        raise ValueError("saved control identity coverage mismatch")
    aligned = frame.iloc[indexer].reset_index(drop=True)
    if not np.array_equal(
        aligned["well_id"].astype(str).to_numpy(),
        bank.keys["well"].astype(str).to_numpy(),
    ):
        raise ValueError("saved control well identity mismatch")
    if not np.array_equal(
        aligned["row_idx"].to_numpy(dtype=np.int64),
        bank.keys["well_row_idx"].to_numpy(dtype=np.int64),
    ):
        raise ValueError("saved control row identity mismatch")
    folds, fold_by_well, bank_fold_match_fraction = extract_saved_control_folds(
        aligned,
        bank,
        int(get_nested(config, "validation.n_folds")),
    )
    values = pd.to_numeric(
        aligned["tvt_pred"], errors="raise"
    ).to_numpy(dtype=np.float64)
    bank_control = np.asarray(
        bank.values[:, bank.candidate_ids.index("exp226_k16")],
        dtype=np.float64,
    )
    parity = float(np.max(np.abs(values - bank_control), initial=0.0))
    return SavedControl(
        path=path,
        values=values,
        frame=aligned,
        file_sha256=sha256_file(path),
        decompressed_sha256=sha256_decompressed_gzip(path),
        parity_max_abs_ft=parity,
        folds=folds,
        fold_by_well=fold_by_well,
        bank_fold_match_fraction=bank_fold_match_fraction,
    )


@dataclass(frozen=True)
class FreezeEvidence:
    contract_path: Path
    contract_file_sha256: str
    manifest_path: Path
    manifest_file_sha256: str
    block_path: Path
    block_file_sha256: str
    block_decompressed_sha256: str
    input_manifest_path: Path
    raw_input_manifest_path: Path
    prediction_paths: dict[str, Path]
    kappa_paths: dict[str, Path]
    prediction_file_sha256: dict[str, str]
    prediction_decompressed_sha256: dict[str, str]
    prediction_content_sha256: dict[str, str]
    candidate_bank_content_sha256: str
    raw_input_content_sha256: str
    evaluation_truth_access_count_before_freeze: int


def build_contract_payload(config: Mapping[str, Any]) -> dict[str, Any]:
    return {
        "experiment": EXPERIMENT_NAME,
        "created_at": datetime.now(UTC).isoformat(),
        "route": get_nested(config, "experiment.route"),
        "parent": get_nested(config, "lineage.parent"),
        "variants": get_nested(config, "model.params.variants"),
        "added_candidate": get_nested(config, "validation.added_candidate"),
        "candidate_order": list(get_nested(config, "candidate_novelty_audit.candidate_order")),
        "block_horizons_rows": get_nested(
            config, "candidate_novelty_audit.block_horizons_rows"
        ),
        "success_criteria": get_nested(config, "success_criteria"),
        "execution": get_nested(config, "execution"),
        "forbidden_actions": get_nested(config, "forbidden_actions"),
    }


def freeze_target_free_bundle(
    bank: CandidateBank,
    assignments: BlockAssignments,
    control: SavedControl,
    variants: Mapping[str, VariantPrediction],
    raw_manifest: pd.DataFrame,
    hidden_evidence: Mapping[str, Any],
    config: Mapping[str, Any],
    artifacts_dir: Path,
) -> FreezeEvidence:
    contract_path = artifacts_dir / f"{OUTPUT_PREFIX}_contract.json"
    write_json(contract_path, build_contract_payload(config))

    raw_manifest_path = artifacts_dir / f"{OUTPUT_PREFIX}_raw_input_manifest.csv"
    _write_frame(raw_manifest_path, raw_manifest)
    raw_input_content_sha = frame_content_sha256(raw_manifest)

    target_free_input_manifest = pd.DataFrame(
        [
            *bank.input_evidence,
            {
                "phase": "target_free_saved_control",
                "source": "exp226_saved_k16_oof_allowlist",
                "path": str(control.path),
                "rows": len(control.frame),
                "file_sha256": control.file_sha256,
                "decompressed_content_sha256": control.decompressed_sha256,
                "logical_content_sha256": frame_content_sha256(
                    control.frame,
                    ["well_id", "row_idx", "suffix_offset", "tvt_pred", "fold"],
                ),
                "schema_sha256": frame_schema_sha256(
                    control.frame[
                        ["well_id", "row_idx", "suffix_offset", "tvt_pred", "fold"]
                    ]
                ),
            },
            *[
                dict(bundle.source_evidence)
                for bundle in variants.values()
                if bundle.source_evidence is not None
            ],
            dict(hidden_evidence),
            {
                "phase": "target_free_generation_input",
                "source": "raw_horizontal_and_typewell_manifest",
                "path": str(raw_manifest_path),
                "rows": len(raw_manifest),
                "file_sha256": sha256_file(raw_manifest_path),
                "decompressed_content_sha256": None,
                "logical_content_sha256": raw_input_content_sha,
                "schema_sha256": frame_schema_sha256(raw_manifest),
            },
        ]
    )
    input_manifest_path = artifacts_dir / f"{OUTPUT_PREFIX}_input_manifest.csv"
    _write_frame(input_manifest_path, target_free_input_manifest)

    block_path = artifacts_dir / f"{OUTPUT_PREFIX}_block_assignment.csv.gz"
    block_file_sha, block_decompressed_sha = write_gzip_frame(
        block_path, assignments.frame
    )

    prediction_paths: dict[str, Path] = {}
    prediction_file_sha: dict[str, str] = {}
    prediction_decompressed_sha: dict[str, str] = {}
    prediction_content_sha: dict[str, str] = {}
    kappa_paths: dict[str, Path] = {}
    kappa_evidence: dict[str, Any] = {}
    for variant, bundle in variants.items():
        prediction_path = (
            artifacts_dir / f"{OUTPUT_PREFIX}_{variant}_oof_predictions.csv.gz"
        )
        file_sha, decompressed_sha = write_gzip_frame(
            prediction_path, bundle.frame
        )
        kappa_path = (
            artifacts_dir / f"{OUTPUT_PREFIX}_{variant}_kappa_by_fold.csv"
        )
        _write_frame(kappa_path, bundle.kappa_by_fold)
        prediction_paths[variant] = prediction_path
        kappa_paths[variant] = kappa_path
        prediction_file_sha[variant] = file_sha
        prediction_decompressed_sha[variant] = decompressed_sha
        prediction_content_sha[variant] = bundle.prediction_content_sha256
        kappa_evidence[variant] = {
            "path": str(kappa_path),
            "file_sha256": sha256_file(kappa_path),
            "logical_content_sha256": frame_content_sha256(bundle.kappa_by_fold),
        }

    manifest = {
        "experiment": EXPERIMENT_NAME,
        "status": "target_free_predictions_bank_and_blocks_frozen",
        "frozen_at": datetime.now(UTC).isoformat(),
        "rows": len(bank.keys),
        "wells": int(bank.keys["well"].nunique()),
        "candidate_bank_folds": sorted(
            int(value) for value in bank.keys["outer_fold"].unique()
        ),
        "evaluation_folds": sorted(int(value) for value in np.unique(control.folds)),
        "variants": {
            variant: {
                "k_segments": bundle.k_segments,
                "rows": len(bundle.frame),
                "finite_coverage": float(np.isfinite(bundle.values).mean()),
                "prediction_path": str(prediction_paths[variant]),
                "prediction_file_sha256": prediction_file_sha[variant],
                "prediction_decompressed_sha256": prediction_decompressed_sha[variant],
                "prediction_content_sha256": prediction_content_sha[variant],
                "donor_fit_truth_wells": bundle.donor_fit_truth_wells,
                "fold_run_count": bundle.fold_run_count,
                "outer_valid_truth_state_count": 0,
                "source_evidence": bundle.source_evidence,
            }
            for variant, bundle in variants.items()
        },
        "saved_control": {
            "path": str(control.path),
            "file_sha256": control.file_sha256,
            "decompressed_content_sha256": control.decompressed_sha256,
            "parity_max_abs_ft_vs_exp293_bank": control.parity_max_abs_ft,
            "bank_fold_match_fraction": control.bank_fold_match_fraction,
            "fold_role": "exp226_evaluation_fold",
            "regenerated": False,
        },
        "exp293_bank": {
            "candidate_count": len(bank.candidate_ids),
            "candidate_order": list(bank.candidate_ids),
            "candidate_content_sha256": bank.candidate_content_sha256,
            "expected_candidate_content_sha256": get_nested(
                config, "data.exp293_bank.expected_candidate_content_sha256"
            ),
        },
        "block_assignment": {
            "path": str(block_path),
            "file_sha256": block_file_sha,
            "decompressed_content_sha256": block_decompressed_sha,
            "logical_content_sha256": frame_content_sha256(assignments.frame),
            "expected_decompressed_sha256": get_nested(
                config, "data.exp293_bank.expected_block_assignment_decompressed_sha256"
            ),
            "outer_fold_role": "exp263_bank_provenance_and_exp293_sha_only",
        },
        "raw_input_manifest": {
            "path": str(raw_manifest_path),
            "rows": len(raw_manifest),
            "file_sha256": sha256_file(raw_manifest_path),
            "logical_content_sha256": raw_input_content_sha,
        },
        "target_free_input_manifest": {
            "path": str(input_manifest_path),
            "rows": len(target_free_input_manifest),
            "file_sha256": sha256_file(input_manifest_path),
            "logical_content_sha256": frame_content_sha256(
                target_free_input_manifest
            ),
        },
        "hidden_like_assignment": dict(hidden_evidence),
        "kappa_by_fold": kappa_evidence,
        "evaluation_truth_access_count_before_freeze": 0,
        "evaluation_truth_columns_loaded_before_freeze": [],
        "candidate_truth_policy": (
            "The saved exp333 gzip physically contains tvt_true, but the "
            "pre-freeze loader opens only the explicit target-free usecols."
        ),
        "contract_file_sha256": sha256_file(contract_path),
        "config_file_sha256": sha256_file(find_config_path()),
        "frozen": True,
    }
    manifest_path = artifacts_dir / f"{OUTPUT_PREFIX}_freeze_manifest.json"
    write_json(manifest_path, manifest)
    return FreezeEvidence(
        contract_path=contract_path,
        contract_file_sha256=sha256_file(contract_path),
        manifest_path=manifest_path,
        manifest_file_sha256=sha256_file(manifest_path),
        block_path=block_path,
        block_file_sha256=block_file_sha,
        block_decompressed_sha256=block_decompressed_sha,
        input_manifest_path=input_manifest_path,
        raw_input_manifest_path=raw_manifest_path,
        prediction_paths=prediction_paths,
        kappa_paths=kappa_paths,
        prediction_file_sha256=prediction_file_sha,
        prediction_decompressed_sha256=prediction_decompressed_sha,
        prediction_content_sha256=prediction_content_sha,
        candidate_bank_content_sha256=bank.candidate_content_sha256,
        raw_input_content_sha256=raw_input_content_sha,
        evaluation_truth_access_count_before_freeze=0,
    )


def verify_freeze_before_truth(
    bank: CandidateBank,
    variants: Mapping[str, VariantPrediction],
    freeze: FreezeEvidence,
    config: Mapping[str, Any],
) -> None:
    if freeze.evaluation_truth_access_count_before_freeze != 0:
        raise ValueError("evaluation truth was accessed before freeze")
    if sha256_file(freeze.contract_path) != freeze.contract_file_sha256:
        raise ValueError("frozen contract changed before truth load")
    if sha256_file(freeze.manifest_path) != freeze.manifest_file_sha256:
        raise ValueError("freeze manifest changed before truth load")
    if sha256_file(freeze.block_path) != freeze.block_file_sha256:
        raise ValueError("block assignment changed before truth load")
    if (
        sha256_decompressed_gzip(freeze.block_path)
        != freeze.block_decompressed_sha256
    ):
        raise ValueError("block assignment decompressed content changed")
    if freeze.block_decompressed_sha256 != str(
        get_nested(
            config,
            "data.exp293_bank.expected_block_assignment_decompressed_sha256",
        )
    ):
        raise ValueError("block assignment does not match completed exp293")
    current_bank_sha = candidate_bank_content_sha256(
        bank, int(get_nested(config, "candidate_novelty_audit.work_chunk_rows"))
    )
    expected_bank_sha = str(
        get_nested(config, "data.exp293_bank.expected_candidate_content_sha256")
    )
    if current_bank_sha != freeze.candidate_bank_content_sha256:
        raise ValueError("candidate bank changed after freeze")
    if current_bank_sha != expected_bank_sha:
        raise ValueError("candidate bank does not match completed exp293")
    for variant, bundle in variants.items():
        if (
            frame_content_sha256(
                bundle.frame,
                [
                    "id",
                    "well",
                    "well_row_idx",
                    "outer_fold",
                    "variant",
                    "candidate_tvt",
                ],
            )
            != freeze.prediction_content_sha256[variant]
        ):
            raise ValueError(f"{variant} in-memory prediction changed after freeze")
        if (
            sha256_file(freeze.prediction_paths[variant])
            != freeze.prediction_file_sha256[variant]
        ):
            raise ValueError(f"{variant} prediction file changed after freeze")
        if (
            sha256_decompressed_gzip(freeze.prediction_paths[variant])
            != freeze.prediction_decompressed_sha256[variant]
        ):
            raise ValueError(f"{variant} decompressed prediction changed after freeze")

## 6. Post-freeze truth loader and direct readout

In [ ]:
def load_truth_after_freeze(
    bank: CandidateBank,
    variants: Mapping[str, VariantPrediction],
    freeze: FreezeEvidence,
    config: Mapping[str, Any],
) -> tuple[np.ndarray, list[dict[str, Any]], str]:
    verify_freeze_before_truth(bank, variants, freeze, config)
    expected_wells = set(bank.keys["well"].astype(str))
    raw_dir, files = resolve_raw_train_dir(config, expected_wells)
    truth_column = str(get_nested(config, "data.raw_columns.truth"))
    visible_column = str(get_nested(config, "data.raw_columns.visible_input"))
    truth_frames: list[pd.DataFrame] = []
    input_evidence: list[dict[str, Any]] = []
    for path in files:
        well = path.name.split("__horizontal_well.csv", 1)[0]
        frame = pd.read_csv(path, usecols=[truth_column, visible_column])
        visible = pd.to_numeric(frame[visible_column], errors="coerce")
        suffix_mask = visible.isna().to_numpy()
        row_idx = np.flatnonzero(suffix_mask).astype(np.int32)
        truth_values = pd.to_numeric(
            frame.loc[suffix_mask, truth_column], errors="raise"
        ).to_numpy(dtype=np.float64)
        if not np.isfinite(truth_values).all():
            raise ValueError(f"raw truth contains nonfinite suffix TVT: {path}")
        truth_frames.append(
            pd.DataFrame(
                {
                    "id": [
                        f"{well}_{int(index)}" for index in row_idx
                    ],
                    "well": well,
                    "well_row_idx": row_idx,
                    "true_tvt": truth_values,
                }
            )
        )
        input_evidence.append(
            {
                "phase": "post_freeze_evaluation_truth",
                "source": "raw_train_horizontal",
                "path": str(path),
                "rows": len(frame),
                "suffix_rows": len(row_idx),
                "file_sha256": sha256_file(path),
                "raw_train_dir": str(raw_dir),
            }
        )
    truth_frame = pd.concat(truth_frames, ignore_index=True)
    if truth_frame["id"].duplicated().any() or len(truth_frame) != len(bank.keys):
        raise ValueError("post-freeze truth identity/count contract failed")
    indexer = pd.Index(truth_frame["id"].astype(str)).get_indexer(
        bank.keys["id"].astype(str)
    )
    if np.any(indexer < 0):
        raise ValueError("candidate IDs are missing from post-freeze truth")
    aligned = truth_frame.iloc[indexer].reset_index(drop=True)
    if not np.array_equal(
        aligned["id"].astype(str).to_numpy(),
        bank.keys["id"].astype(str).to_numpy(),
    ):
        raise ValueError("post-freeze truth alignment failed")
    truth = aligned["true_tvt"].to_numpy(dtype=np.float64)
    return truth, input_evidence, frame_content_sha256(
        aligned[["id", "true_tvt"]]
    )


def rmse_from_arrays(truth: np.ndarray, prediction: np.ndarray) -> float:
    return float(np.sqrt(np.mean(np.square(prediction - truth))))


def direct_metric_record(
    variant: str,
    scope: str,
    fold: int | None,
    mask: np.ndarray,
    truth: np.ndarray,
    prediction: np.ndarray,
) -> dict[str, Any]:
    rows = int(mask.sum())
    if rows == 0:
        return {
            "variant": variant,
            "scope": scope,
            "fold": fold,
            "rows": 0,
            "rmse": math.nan,
            "mae": math.nan,
            "bias": math.nan,
            "within10": math.nan,
        }
    error = prediction[mask] - truth[mask]
    return {
        "variant": variant,
        "scope": scope,
        "fold": fold,
        "rows": rows,
        "rmse": float(np.sqrt(np.mean(np.square(error)))),
        "mae": float(np.mean(np.abs(error))),
        "bias": float(np.mean(error)),
        "within10": float(np.mean(np.abs(error) <= 10.0)),
    }


def build_direct_readouts(
    bank: CandidateBank,
    control: SavedControl,
    variants: Mapping[str, VariantPrediction],
    truth: np.ndarray,
    hidden_sets: Mapping[str, set[str]],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    predictions = {
        "exp226_k16": control.values,
        **{name: bundle.values for name, bundle in variants.items()},
    }
    folds = control.folds
    wells = bank.keys["well"].astype(str).to_numpy()
    md_since = bank.keys["md_since"].to_numpy(dtype=np.float64)
    records: list[dict[str, Any]] = []
    all_rows = np.ones(len(bank.keys), dtype=bool)
    for name, prediction in predictions.items():
        records.append(
            direct_metric_record(
                name, "pooled", None, all_rows, truth, prediction
            )
        )
        for fold in range(5):
            mask = folds == fold
            records.append(
                direct_metric_record(
                    name, "fold", fold, mask, truth, prediction
                )
            )
        for scope, (lo, hi) in DIRECT_BUCKETS.items():
            mask = (md_since >= lo) & (md_since < hi)
            records.append(
                direct_metric_record(
                    name, scope, None, mask, truth, prediction
                )
            )
        for scope, selected_wells in hidden_sets.items():
            mask = np.isin(wells, np.asarray(sorted(selected_wells), dtype=object))
            records.append(
                direct_metric_record(
                    name, scope, None, mask, truth, prediction
                )
            )

    well_codes, well_names = pd.factorize(wells, sort=False)
    if not np.array_equal(well_names.astype(str), pd.unique(wells)):
        raise ValueError("well factorization order changed")
    by_well_records: list[dict[str, Any]] = []
    for name, prediction in predictions.items():
        error_squared = np.square(prediction - truth)
        sse = np.bincount(well_codes, weights=error_squared)
        rows = np.bincount(well_codes)
        well_fold = control.fold_by_well
        for position, well in enumerate(well_names.astype(str)):
            by_well_records.append(
                {
                    "variant": name,
                    "well": well,
                    "outer_fold": int(well_fold[well]),
                    "rows": int(rows[position]),
                    "rmse": float(np.sqrt(sse[position] / rows[position])),
                }
            )
    return pd.DataFrame(records), pd.DataFrame(by_well_records)


def one_direct_metric(
    metrics: pd.DataFrame,
    variant: str,
    scope: str,
    fold: int | None = None,
) -> float:
    selected = metrics[
        metrics["variant"].eq(variant) & metrics["scope"].eq(scope)
    ]
    if fold is None:
        selected = selected[selected["fold"].isna()]
    else:
        selected = selected[selected["fold"].eq(fold)]
    if len(selected) != 1:
        raise ValueError(
            f"direct metric not unique: {variant}/{scope}/{fold}"
        )
    return float(selected.iloc[0]["rmse"])


def evaluate_direct_guards(
    direct_metrics: pd.DataFrame,
    by_well: pd.DataFrame,
    config: Mapping[str, Any],
) -> dict[str, Any]:
    """Record direct quality as context without making it a promotion gate."""
    context = get_nested(config, "success_criteria.direct_context_only")
    control = "exp226_k16"
    output: dict[str, Any] = {}
    for variant in EXPECTED_VARIANTS:
        fold_deltas = [
            one_direct_metric(direct_metrics, variant, "fold", fold)
            - one_direct_metric(direct_metrics, control, "fold", fold)
            for fold in range(5)
        ]
        scope_deltas = {
            scope: one_direct_metric(direct_metrics, variant, scope)
            - one_direct_metric(direct_metrics, control, scope)
            for scope in (
                "1000_plus",
                "hidden_like_spatial",
                "hidden_like_typewell_purged",
            )
        }
        variant_well = by_well[by_well["variant"].eq(variant)]["rmse"]
        control_well = by_well[by_well["variant"].eq(control)]["rmse"]
        p95_delta = float(
            variant_well.quantile(0.95) - control_well.quantile(0.95)
        )
        worst_delta = float(variant_well.max() - control_well.max())
        pooled_rmse = one_direct_metric(direct_metrics, variant, "pooled")
        pooled_delta = pooled_rmse - one_direct_metric(
            direct_metrics, control, "pooled"
        )
        parity_checks = {
            "expected_exp333_rmse": abs(
                pooled_rmse - float(context["expected_exp333_rmse_ft"])
            )
            <= float(
                get_nested(
                    config,
                    "success_criteria.technical_all_required."
                    "required_exp333_direct_rmse_parity_max_abs_ft",
                )
            ),
            "expected_exp333_minus_exp226": abs(
                pooled_delta - float(context["expected_exp333_minus_exp226_ft"])
            )
            <= 0.001,
        }
        output[variant] = {
            "passed": False,
            "gate_enabled": False,
            "parity_checks": parity_checks,
            "pooled_rmse": pooled_rmse,
            "pooled_delta_ft": pooled_delta,
            "fold_deltas_ft": fold_deltas,
            "improved_folds": int(np.sum(np.asarray(fold_deltas) < 0.0)),
            "scope_deltas_ft": scope_deltas,
            "by_well_p95_delta_ft": p95_delta,
            "worst_well_delta_ft": worst_delta,
            "exp228_reference_rmse_ft": float(
                context["exp228_reference_rmse_ft"]
            ),
            "exp263_reference_rmse_ft": float(
                context["exp263_reference_rmse_ft"]
            ),
        }
    return {
        "variants": output,
        "passed": False,
        "gate_enabled": False,
        "all_parity_checks_passed": bool(
            all(
                all(item["parity_checks"].values())
                for item in output.values()
            )
        ),
    }

## 7. Add-one candidate novelty readout

In [ ]:
@dataclass
class OracleReference:
    row_best_sse: np.ndarray
    group_sse: dict[str, np.ndarray]
    candidate_total_sse: np.ndarray


def compute_oracle_reference(
    bank: CandidateBank,
    assignments: BlockAssignments,
    truth: np.ndarray,
) -> OracleReference:
    n_candidates = len(bank.candidate_ids)
    row_best = np.full(len(truth), np.inf, dtype=np.float64)
    candidate_total = np.zeros(n_candidates, dtype=np.float64)
    group_sse = {
        name: np.zeros((layout.n_groups, n_candidates), dtype=np.float64)
        for name, layout in assignments.layouts.items()
    }
    for position in range(n_candidates):
        prediction = np.asarray(bank.values[:, position], dtype=np.float64)
        error_squared = np.square(prediction - truth)
        row_best = np.minimum(row_best, error_squared)
        candidate_total[position] = float(error_squared.sum())
        for name, layout in assignments.layouts.items():
            group_sse[name][:, position] = np.bincount(
                layout.codes,
                weights=error_squared,
                minlength=layout.n_groups,
            )
    return OracleReference(
        row_best_sse=row_best,
        group_sse=group_sse,
        candidate_total_sse=candidate_total,
    )


def pearson_correlation(left: np.ndarray, right: np.ndarray) -> float:
    left_centered = left - float(np.mean(left))
    right_centered = right - float(np.mean(right))
    denominator = float(
        np.sqrt(
            np.dot(left_centered, left_centered)
            * np.dot(right_centered, right_centered)
        )
    )
    if denominator <= 0.0:
        return math.nan
    return float(np.dot(left_centered, right_centered) / denominator)


def novelty_metric_record(
    *,
    variant: str,
    granularity: str,
    scope: str,
    fold: int | None,
    group_mask: np.ndarray,
    group_rows: np.ndarray,
    base_best_sse: np.ndarray,
    added_sse: np.ndarray,
    tie_atol: float,
) -> dict[str, Any]:
    selected = group_mask & (group_rows > 0)
    rows = int(group_rows[selected].sum())
    groups = int(selected.sum())
    base = base_best_sse[selected]
    added = added_sse[selected]
    improved = added + tie_atol < base
    oracle = np.where(improved, added, base)
    return {
        "variant": variant,
        "granularity": granularity,
        "scope": scope,
        "fold": fold,
        "rows": rows,
        "groups": groups,
        "base_oracle_rmse": float(np.sqrt(base.sum() / rows)),
        "add_one_oracle_rmse": float(np.sqrt(oracle.sum() / rows)),
        "oracle_improvement_ft": float(
            np.sqrt(base.sum() / rows) - np.sqrt(oracle.sum() / rows)
        ),
        "strict_unique_best_groups": int(improved.sum()),
        "strict_unique_best_fraction": float(improved.mean()),
        "strict_unique_best_rows": int(group_rows[selected][improved].sum()),
    }


def build_novelty_readouts(
    bank: CandidateBank,
    assignments: BlockAssignments,
    reference: OracleReference,
    variants: Mapping[str, VariantPrediction],
    truth: np.ndarray,
    tie_atol: float,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    metric_records: list[dict[str, Any]] = []
    correlation_records: list[dict[str, Any]] = []
    by_well_records: list[dict[str, Any]] = []
    row_mask = np.ones(len(truth), dtype=bool)
    row_rows = np.ones(len(truth), dtype=np.int64)
    row_folds = bank.keys["outer_fold"].to_numpy(dtype=np.int8)
    for variant, bundle in variants.items():
        added_values = bundle.values.astype(np.float32).astype(np.float64)
        added_row_sse = np.square(added_values - truth)
        metric_records.append(
            novelty_metric_record(
                variant=variant,
                granularity="row",
                scope="pooled",
                fold=None,
                group_mask=row_mask,
                group_rows=row_rows,
                base_best_sse=reference.row_best_sse,
                added_sse=added_row_sse,
                tie_atol=tie_atol,
            )
        )
        for position, candidate in enumerate(bank.candidate_ids):
            correlation_records.append(
                {
                    "variant": variant,
                    "bank_candidate": candidate,
                    "pearson_prediction_correlation": pearson_correlation(
                        added_values,
                        np.asarray(bank.values[:, position], dtype=np.float64),
                    ),
                }
            )
        for name, layout in assignments.layouts.items():
            added_group_sse = np.bincount(
                layout.codes,
                weights=added_row_sse,
                minlength=layout.n_groups,
            )
            base_best = np.min(reference.group_sse[name], axis=1)
            all_groups = np.ones(layout.n_groups, dtype=bool)
            metric_records.append(
                novelty_metric_record(
                    variant=variant,
                    granularity=name,
                    scope="pooled",
                    fold=None,
                    group_mask=all_groups,
                    group_rows=layout.group_rows,
                    base_best_sse=base_best,
                    added_sse=added_group_sse,
                    tie_atol=tie_atol,
                )
            )
            for fold in range(5):
                metric_records.append(
                    novelty_metric_record(
                        variant=variant,
                        granularity=name,
                        scope="fold",
                        fold=fold,
                        group_mask=layout.group_fold == fold,
                        group_rows=layout.group_rows,
                        base_best_sse=base_best,
                        added_sse=added_group_sse,
                        tie_atol=tie_atol,
                    )
                )
            if name == "h512":
                strict = added_group_sse + tie_atol < base_best
                for well_code, well in enumerate(assignments.well_names.astype(str)):
                    selected = layout.group_well == well_code
                    by_well_records.append(
                        {
                            "variant": variant,
                            "well": well,
                            "outer_fold": int(assignments.well_fold[well_code]),
                            "h512_blocks": int(selected.sum()),
                            "h512_strict_unique_best_blocks": int(
                                strict[selected].sum()
                            ),
                            "h512_strict_unique_best_fraction": float(
                                strict[selected].mean()
                            ),
                        }
                    )
        for fold in range(5):
            selected = row_folds == fold
            metric_records.append(
                novelty_metric_record(
                    variant=variant,
                    granularity="row",
                    scope="fold",
                    fold=fold,
                    group_mask=selected,
                    group_rows=row_rows,
                    base_best_sse=reference.row_best_sse,
                    added_sse=added_row_sse,
                    tie_atol=tie_atol,
                )
            )
    return (
        pd.DataFrame(metric_records),
        pd.DataFrame(correlation_records),
        pd.DataFrame(by_well_records),
    )


def one_novelty_metric(
    metrics: pd.DataFrame,
    variant: str,
    granularity: str,
    scope: str,
    fold: int | None = None,
) -> pd.Series:
    selected = metrics[
        metrics["variant"].eq(variant)
        & metrics["granularity"].eq(granularity)
        & metrics["scope"].eq(scope)
    ]
    if fold is None:
        selected = selected[selected["fold"].isna()]
    else:
        selected = selected[selected["fold"].eq(fold)]
    if len(selected) != 1:
        raise ValueError(
            f"novelty metric not unique: {variant}/{granularity}/{scope}/{fold}"
        )
    return selected.iloc[0]


def evaluate_novelty_guards(
    novelty_metrics: pd.DataFrame, config: Mapping[str, Any]
) -> dict[str, Any]:
    criteria = get_nested(
        config,
        "success_criteria.candidate_novelty_pass_for_at_least_one_variant",
    )
    output: dict[str, Any] = {}
    for variant in EXPECTED_VARIANTS:
        h512 = one_novelty_metric(
            novelty_metrics, variant, "h512", "pooled"
        )
        whole = one_novelty_metric(
            novelty_metrics, variant, "whole_well", "pooled"
        )
        fold_improvements = [
            float(
                one_novelty_metric(
                    novelty_metrics, variant, "h512", "fold", fold
                )["oracle_improvement_ft"]
            )
            for fold in range(5)
        ]
        checks = {
            "h512_oracle_improvement": float(h512["oracle_improvement_ft"])
            >= float(criteria["minimum_h512_oracle_rmse_improvement_ft"]),
            "whole_well_oracle_improvement": float(
                whole["oracle_improvement_ft"]
            )
            >= float(criteria["minimum_whole_well_oracle_rmse_improvement_ft"]),
            "h512_strict_unique_best_fraction": float(
                h512["strict_unique_best_fraction"]
            )
            >= float(criteria["minimum_h512_strict_unique_best_fraction"]),
            "improved_folds": int(np.sum(np.asarray(fold_improvements) > 0.0))
            >= int(criteria["minimum_improved_folds"]),
        }
        output[variant] = {
            "passed": bool(all(checks.values())),
            "checks": checks,
            "h512_base_oracle_rmse": float(h512["base_oracle_rmse"]),
            "h512_add_one_oracle_rmse": float(h512["add_one_oracle_rmse"]),
            "h512_oracle_improvement_ft": float(h512["oracle_improvement_ft"]),
            "whole_well_oracle_improvement_ft": float(
                whole["oracle_improvement_ft"]
            ),
            "h512_strict_unique_best_fraction": float(
                h512["strict_unique_best_fraction"]
            ),
            "h512_fold_improvements_ft": fold_improvements,
            "improved_folds": int(np.sum(np.asarray(fold_improvements) > 0.0)),
        }
    return {
        "variants": output,
        "passed": bool(any(item["passed"] for item in output.values())),
    }

## 8. PASS/FAIL decision and generated artifacts

In [ ]:
def validate_execution_contract(
    config: Mapping[str, Any], *, require_kaggle_authorization: bool
) -> None:
    if get_nested(config, "experiment.route") != "ensemble":
        raise ValueError("exp361 route must remain ensemble")
    if get_nested(config, "experiment.status") not in {
        "implementation_complete_not_executed",
        "kaggle_cpu_v2_completed_technical_pass_candidate_novelty_pass",
    }:
        raise ValueError("exp361 status differs from an allowed lifecycle state")
    if not bool(get_nested(config, "execution.implementation")):
        raise ValueError("exp361 implementation flag is disabled")
    if not bool(get_nested(config, "execution.implementation_authorized")):
        raise ValueError("exp361 implementation is not authorized")
    if require_kaggle_authorization and not bool(
        get_nested(config, "execution.kaggle_execution_authorized")
    ):
        raise ValueError("exp361 Kaggle execution is not authorized")
    variants = [
        str(item["name"])
        for item in get_nested(config, "model.params.variants")
    ]
    if variants != list(EXPECTED_VARIANTS):
        raise ValueError("exp361 must contain exactly the saved exp333 candidate")
    execution = get_nested(config, "execution")
    expected = {
        "active_scientific_candidates": 1,
        "outer_reporting_folds": 5,
        "total_candidate_fold_readouts": 5,
        "lightgbm_config_count": 0,
        "trained_fold_count": 0,
        "total_boosters": 0,
    }
    for key, value in expected.items():
        if int(execution[key]) != value:
            raise ValueError(f"execution contract changed: {key}")
    if bool(execution["parent_or_control_regeneration"]):
        raise ValueError("parent/control regeneration is forbidden")
    if bool(execution["gpu"]) or bool(execution["inference"]) or bool(
        execution["submission"]
    ):
        raise ValueError("GPU, inference, and submission must remain disabled")


def evaluate_technical_checks(
    bank: CandidateBank,
    assignments: BlockAssignments,
    control: SavedControl,
    variants: Mapping[str, VariantPrediction],
    freeze: FreezeEvidence,
    direct_metrics: pd.DataFrame,
    config: Mapping[str, Any],
) -> dict[str, Any]:
    expected_rows = int(get_nested(config, "validation.expected_rows"))
    expected_wells = int(get_nested(config, "validation.expected_wells"))
    expected_bank_sha = str(
        get_nested(config, "data.exp293_bank.expected_candidate_content_sha256")
    )
    expected_block_sha = str(
        get_nested(
            config,
            "data.exp293_bank.expected_block_assignment_decompressed_sha256",
        )
    )
    control_rmse = one_direct_metric(direct_metrics, "exp226_k16", "pooled")
    checks = {
        "row_count_exact": len(bank.keys) == expected_rows,
        "well_count_exact": int(bank.keys["well"].nunique()) == expected_wells,
        "duplicate_rows_zero": not bank.keys["id"].duplicated().any(),
        "fold_inventory_exact": set(bank.keys["outer_fold"].astype(int))
        == set(range(5)),
        "saved_control_evaluation_fold_inventory_exact": set(
            np.unique(control.folds).astype(int)
        )
        == set(range(5)),
        "candidate_bank_sha_match": bank.candidate_content_sha256
        == expected_bank_sha,
        "block_assignment_sha_match": freeze.block_decompressed_sha256
        == expected_block_sha,
        "control_decompressed_sha_match": control.decompressed_sha256
        == str(
            get_nested(config, "data.exp226_oof.expected_decompressed_sha256")
        ),
        "control_bank_parity": control.parity_max_abs_ft
        <= float(
            get_nested(
                config,
                "success_criteria.technical_all_required.required_control_parity_max_abs_ft",
            )
        ),
        "control_rmse_parity": abs(
            control_rmse
            - float(get_nested(config, "validation.control_oof_rmse_ft"))
        )
        <= 0.001,
        "variant_inventory_exact": {
            name: bundle.k_segments for name, bundle in variants.items()
        }
        == EXPECTED_VARIANTS,
        "variant_finite_coverage_one": all(
            np.isfinite(bundle.values).all() for bundle in variants.values()
        ),
        "variant_row_count_exact": all(
            len(bundle.values) == expected_rows for bundle in variants.values()
        ),
        "variant_fold_runs_exact": all(
            bundle.fold_run_count == 5 for bundle in variants.values()
        ),
        "exp333_source_decompressed_sha_match": all(
            bundle.source_evidence is not None
            and bundle.source_evidence["decompressed_content_sha256"]
            == str(
                get_nested(
                    config, "data.exp333_oof.expected_decompressed_sha256"
                )
            )
            for bundle in variants.values()
        ),
        "exp333_upstream_prediction_sha_recorded": all(
            bundle.source_evidence is not None
            and bundle.source_evidence["upstream_reported_prediction_sha256"]
            == str(
                get_nested(
                    config,
                    "data.exp333_oof.upstream_reported_prediction_sha256",
                )
            )
            for bundle in variants.values()
        ),
        "exp333_direct_metric_parity": abs(
            one_direct_metric(
                direct_metrics, "exp333_segment_offset", "pooled"
            )
            - float(
                get_nested(
                    config,
                    "validation.expected_added_candidate_oof_rmse_ft",
                )
            )
        )
        <= float(
            get_nested(
                config,
                "success_criteria.technical_all_required."
                "required_exp333_direct_rmse_parity_max_abs_ft",
            )
        ),
        "exp333_outer_fold_parity": all(
            np.array_equal(
                bundle.frame["outer_fold"].to_numpy(dtype=np.int8),
                control.folds,
            )
            for bundle in variants.values()
        ),
        "evaluation_truth_access_before_freeze_zero": (
            freeze.evaluation_truth_access_count_before_freeze == 0
        ),
        "parent_control_regeneration_zero": not bool(
            get_nested(config, "execution.parent_or_control_regeneration")
        ),
        "bank_and_block_shape_consistent": len(assignments.frame) == expected_rows,
    }
    return {"checks": checks, "passed": bool(all(checks.values()))}


def persist_outputs(
    bank: CandidateBank,
    control: SavedControl,
    variants: Mapping[str, VariantPrediction],
    freeze: FreezeEvidence,
    truth_evidence: Sequence[Mapping[str, Any]],
    truth_content_sha256: str,
    direct_metrics: pd.DataFrame,
    direct_by_well: pd.DataFrame,
    novelty_metrics: pd.DataFrame,
    correlations: pd.DataFrame,
    novelty_by_well: pd.DataFrame,
    technical: Mapping[str, Any],
    direct: Mapping[str, Any],
    novelty: Mapping[str, Any],
    config: Mapping[str, Any],
    artifacts_dir: Path,
) -> dict[str, Any]:
    direct_path = artifacts_dir / f"{OUTPUT_PREFIX}_direct_metrics.csv"
    direct_well_path = artifacts_dir / f"{OUTPUT_PREFIX}_direct_by_well.csv"
    novelty_path = artifacts_dir / f"{OUTPUT_PREFIX}_candidate_novelty_metrics.csv"
    correlation_path = artifacts_dir / f"{OUTPUT_PREFIX}_candidate_correlations.csv"
    novelty_well_path = artifacts_dir / f"{OUTPUT_PREFIX}_candidate_novelty_by_well.csv"
    truth_manifest_path = artifacts_dir / f"{OUTPUT_PREFIX}_post_freeze_truth_manifest.csv"
    summary_path = artifacts_dir / f"{OUTPUT_PREFIX}_summary.json"
    sha_manifest_path = artifacts_dir / f"{OUTPUT_PREFIX}_sha_manifest.csv"
    for path, frame in (
        (direct_path, direct_metrics),
        (direct_well_path, direct_by_well),
        (novelty_path, novelty_metrics),
        (correlation_path, correlations),
        (novelty_well_path, novelty_by_well),
        (truth_manifest_path, pd.DataFrame(truth_evidence)),
    ):
        _write_frame(path, frame)

    novelty_pass = bool(novelty["passed"])
    technical_pass = bool(technical["passed"])
    scientific_support = bool(technical_pass and novelty_pass)
    next_action = str(
        get_nested(
            config,
            "success_criteria.decision.pass"
            if scientific_support
            else "success_criteria.decision.fail",
        )
    )
    decision = {
        "technical_passed": technical_pass,
        "direct_gate_enabled": False,
        "direct_context_parity_passed": bool(
            direct["all_parity_checks_passed"]
        ),
        "candidate_novelty_passed": bool(technical_pass and novelty_pass),
        "scientific_support": scientific_support,
        "next_action": next_action,
    }
    summary = {
        "experiment": EXPERIMENT_NAME,
        "status": "audit_completed",
        "route": "ensemble",
        "completed_at": datetime.now(UTC).isoformat(),
        "rows": len(bank.keys),
        "wells": int(bank.keys["well"].nunique()),
        "control_rmse": one_direct_metric(
            direct_metrics, "exp226_k16", "pooled"
        ),
        "technical": technical,
        "direct": direct,
        "candidate_novelty": novelty,
        "decision": decision,
        "truth_content_sha256": truth_content_sha256,
        "candidate_bank_content_sha256": bank.candidate_content_sha256,
        "block_assignment_decompressed_sha256": (
            freeze.block_decompressed_sha256
        ),
        "variant_prediction_content_sha256": {
            name: bundle.prediction_content_sha256
            for name, bundle in variants.items()
        },
        "variant_source_evidence": {
            name: bundle.source_evidence
            for name, bundle in variants.items()
        },
        "oracle_prediction_persisted": False,
        "execution": {
            "scientific_candidates": 1,
            "reporting_folds": 5,
            "candidate_fold_readouts": 5,
            "lightgbm_configs": 0,
            "trained_folds": 0,
            "boosters": 0,
            "parent_control_regeneration": 0,
            "gpu": False,
            "inference": False,
            "submission": False,
        },
    }
    write_json(summary_path, summary)

    artifact_paths = [
        freeze.contract_path,
        freeze.manifest_path,
        freeze.block_path,
        freeze.input_manifest_path,
        freeze.raw_input_manifest_path,
        *freeze.prediction_paths.values(),
        *freeze.kappa_paths.values(),
        direct_path,
        direct_well_path,
        novelty_path,
        correlation_path,
        novelty_well_path,
        truth_manifest_path,
        summary_path,
    ]
    sha_records = []
    for path in artifact_paths:
        sha_records.append(
            {
                "artifact": path.name,
                "bytes": path.stat().st_size,
                "file_sha256": sha256_file(path),
                "decompressed_content_sha256": (
                    sha256_decompressed_gzip(path)
                    if path.suffix == ".gz"
                    else None
                ),
            }
        )
    sha_manifest = pd.DataFrame(sha_records)
    _write_frame(sha_manifest_path, sha_manifest)

    metrics = {
        "experiment": EXPERIMENT_NAME,
        "status": "implementation_complete_audit_executed",
        "route": "ensemble",
        "cv": direct["variants"]["exp333_segment_offset"]["pooled_rmse"],
        "public_lb": None,
        "private_lb": None,
        "technical_passed": technical_pass,
        "direct_gate_enabled": False,
        "direct_context_parity_passed": bool(
            direct["all_parity_checks_passed"]
        ),
        "candidate_novelty_passed": bool(technical_pass and novelty_pass),
        "decision": decision["next_action"],
        "variants": direct["variants"],
        "candidate_novelty": novelty["variants"],
        "prediction_content_sha256": {
            name: bundle.prediction_content_sha256
            for name, bundle in variants.items()
        },
        "sha_manifest_file_sha256": sha256_file(sha_manifest_path),
    }
    write_json(runtime_metrics_path(), metrics)
    return summary


def run_audit(config: Mapping[str, Any]) -> dict[str, Any]:
    validate_execution_contract(config, require_kaggle_authorization=True)
    artifacts_dir = runtime_artifacts_dir()
    work_dir = runtime_work_dir()
    print("Experiment:", EXPERIMENT_NAME)
    print("Route:", get_nested(config, "experiment.route"))
    print("Parent:", get_nested(config, "lineage.parent"))
    print("Variants:", EXPECTED_VARIANTS)
    print("Execution contract: 1 saved candidate x 5 fold readouts")
    print("LightGBM configs / trained folds / boosters: 0 / 0 / 0")
    print("Parent/control regeneration: 0")

    bank = build_candidate_bank(config, work_dir)
    expected_bank_sha = str(
        get_nested(config, "data.exp293_bank.expected_candidate_content_sha256")
    )
    if bank.candidate_content_sha256 != expected_bank_sha:
        raise ValueError("reconstructed candidate bank SHA differs from exp293")
    horizons = [
        int(value)
        for value in get_nested(
            config, "candidate_novelty_audit.block_horizons_rows"
        )
    ]
    assignments = build_block_assignments(bank.keys, horizons)
    hidden_sets, hidden_evidence = load_hidden_like_sets(
        config, set(bank.keys["well"].astype(str))
    )
    control = load_saved_control(config, bank)
    evaluation_assignments = assignments_with_evaluation_folds(
        assignments, control.folds
    )
    raw_dir, horizontal_files = resolve_raw_train_dir(
        config, set(bank.keys["well"].astype(str))
    )
    raw_manifest = build_raw_input_manifest(raw_dir, horizontal_files)

    variants = {
        variant: build_variant_oof(
            variant,
            config,
            bank,
            horizontal_files,
            control.fold_by_well,
        )
        for variant in EXPECTED_VARIANTS
    }
    freeze = freeze_target_free_bundle(
        bank,
        assignments,
        control,
        variants,
        raw_manifest,
        hidden_evidence,
        config,
        artifacts_dir,
    )
    print("Target-free predictions frozen:", freeze.prediction_content_sha256)
    print("Candidate bank frozen:", freeze.candidate_bank_content_sha256)
    print("Block assignment frozen:", freeze.block_decompressed_sha256)
    print(
        "Evaluation truth access before freeze:",
        freeze.evaluation_truth_access_count_before_freeze,
    )

    truth, truth_evidence, truth_sha = load_truth_after_freeze(
        bank, variants, freeze, config
    )
    direct_metrics, direct_by_well = build_direct_readouts(
        bank, control, variants, truth, hidden_sets
    )
    direct = evaluate_direct_guards(
        direct_metrics, direct_by_well, config
    )
    reference = compute_oracle_reference(bank, assignments, truth)
    novelty_metrics, correlations, novelty_by_well = build_novelty_readouts(
        bank,
        evaluation_assignments,
        reference,
        variants,
        truth,
        float(
            get_nested(
                config,
                "candidate_novelty_audit.tie_atol_squared_ft",
            )
        ),
    )
    novelty = evaluate_novelty_guards(novelty_metrics, config)
    technical = evaluate_technical_checks(
        bank,
        assignments,
        control,
        variants,
        freeze,
        direct_metrics,
        config,
    )
    summary = persist_outputs(
        bank,
        control,
        variants,
        freeze,
        truth_evidence,
        truth_sha,
        direct_metrics,
        direct_by_well,
        novelty_metrics,
        correlations,
        novelty_by_well,
        technical,
        direct,
        novelty,
        config,
        artifacts_dir,
    )
    print("Technical PASS:", technical["passed"])
    print("Direct gate enabled:", direct["gate_enabled"])
    print("Direct context parity PASS:", direct["all_parity_checks_passed"])
    print("Candidate novelty PASS:", novelty["passed"])
    print("Next action:", summary["decision"]["next_action"])
    print("Artifacts:", artifacts_dir)
    return summary

## 9. Setup, contract preview, and execution

In [ ]:
if EXECUTE_NOTEBOOK:
    CONFIG_PATH = find_config_path()
    CONFIG = read_yaml(CONFIG_PATH)
    print("Config:", CONFIG_PATH)
    print("Config SHA256:", sha256_file(CONFIG_PATH))
    validate_execution_contract(CONFIG, require_kaggle_authorization=True)
    print(
        json.dumps(
            {
                "experiment": get_nested(CONFIG, "experiment"),
                "lineage": get_nested(CONFIG, "lineage"),
                "variants": get_nested(CONFIG, "model.params.variants"),
                "added_candidate": get_nested(
                    CONFIG, "validation.added_candidate"
                ),
                "execution": get_nested(CONFIG, "execution"),
                "success_criteria": get_nested(CONFIG, "success_criteria"),
            },
            indent=2,
            ensure_ascii=False,
        )
    )
    SUMMARY = run_audit(CONFIG)
    print(json.dumps(SUMMARY["decision"], indent=2, ensure_ascii=False))